# XGBoost Model - Advanced Churn Prediction

## Objectives
- Build an advanced Gradient Boosting model using XGBoost.
- Perform hyperparameter tuning to optimize performance.
- Evaluate model performance and compare with baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import pickle
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load Data
df = pd.read_csv('../data/processed/final_analytical_dataset.csv')

# Drop non-predictive columns
drop_cols = ['customerID', 'Tenure_Cohort', 'Last_Activity_Date', 'AcquisitionDate', 'CohortMonth']
df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Encode Categorical Variables for XGBoost
label_encoders = {}
for col in df_model.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# Separate Target
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# XGBoost Classifier
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
)

# Hyperparameter Tuning (Simplified Grid Search)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, scoring='roc_auc', verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_

In [ ]:
# Evaluate
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

In [ ]:
# Feature Importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(20), palette='viridis')
plt.title('XGBoost Feature Importance')
plt.show()

In [ ]:
# Save Model
os.makedirs('../models/model_artifacts', exist_ok=True)
with open('../models/model_artifacts/xgb_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)
    
print("Model saved to ../models/model_artifacts/xgb_model.pkl")